In [ ]:
%pip install --quiet -U python-dotenv pydantic-ai scenedetect opencv-python-headless

import os
from pathlib import Path
import nest_asyncio; nest_asyncio.apply()

# Download data files if not already present (e.g. on Colab)
if not Path("data").exists():
    import zipfile, urllib.request
    url = "https://github.com/jsoma/workshop-ai-images-video/raw/main/docs/nicar-2026/07-video-deep-dive-data.zip"
    print("Downloading data...")
    urllib.request.urlretrieve(url, "_data.zip")
    with zipfile.ZipFile("_data.zip") as zf:
        zf.extractall("data")
    Path("_data.zip").unlink()
    print("Done!")

# Paste API keys here or leave empty for .env / Colab secrets
api_keys = {"OPENAI_API_KEY": "", "GOOGLE_API_KEY": ""}
os.environ.update({k: v for k, v in api_keys.items() if v})
try:
    from google.colab import userdata
    for key in api_keys:
        try: os.environ.setdefault(key, userdata.get(key))
        except Exception: pass
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

DATA = Path("data")
Path("outputs").mkdir(exist_ok=True)


# Bonus: Video Deep Dive

The main video notebook showed you how to split video into frames and audio. This one goes deeper: automatic scene detection, and using Gemini to understand video directly — with timestamps, structured data, and zero decomposition.

## Scene detection with PySceneDetect

[PySceneDetect](https://www.scenedetect.com/) finds cuts in video automatically by flagging big visual changes. It's ancient technology, but super quick and (mostly) effective. Great for splitting long videos into meaningful chunks.


**`video/scenes.py`** — Detect scene boundaries and extract mid-scene frames with PySceneDetect


In [ ]:
import cv2
import pandas as pd
from pathlib import Path
from scenedetect import open_video, SceneManager, ContentDetector

DATA = Path("data")
VIDEO = DATA / "rDXubdQdJYs.mp4"
OUTPUT = Path("outputs") / "scenes"
OUTPUT.mkdir(parents=True, exist_ok=True)

video = open_video(str(VIDEO))
scene_manager = SceneManager()
scene_manager.add_detector(ContentDetector(threshold=27.0))
scene_manager.detect_scenes(video)
scene_list = scene_manager.get_scene_list()

rows = []
for i, (start, end) in enumerate(scene_list, 1):
    rows.append({"scene": i, "start_time": start.get_timecode(), "end_time": end.get_timecode(),
                 "duration_sec": round((end - start).get_seconds(), 2)})
df = pd.DataFrame(rows)
df.to_csv(OUTPUT / "scene_list.csv", index=False)

cap = cv2.VideoCapture(str(VIDEO))
fps = cap.get(cv2.CAP_PROP_FPS)
for i, (start, end) in enumerate(scene_list, 1):
    mid_frame = int((start.get_seconds() + end.get_seconds()) / 2 * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)
    ret, frame = cap.read()
    if ret:
        cv2.imwrite(str(OUTPUT / f"scene_{i:03d}.jpg"), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
cap.release()

print(f"Found {len(scene_list)} scenes, frames saved to {OUTPUT}")


objc[914]: Class AVFFrameReceiver is implemented in both /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-nicar/01-fri-analyzing-images-videos-ai/ai-images-video/.venv/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x1053e43a8) and /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-nicar/01-fri-analyzing-images-videos-ai/ai-images-video/.venv/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x11de103a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[914]: Class AVFAudioReceiver is implemented in both /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-nicar/01-fri-analyzing-images-videos-ai/ai-images-video/.venv/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x1053e43f8) and /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-nicar/01-fri-analyzing-images-videos-ai/ai-images-video/.venv/lib/python3.12/site-p

Found 16 scenes, frames saved to outputs/scenes


Each scene gets a start time, end time, and a representative frame saved to disk. Now you can analyze scenes individually instead of processing the whole video.

## Send a video file to Gemini

Gemini can watch video. Upload a file, ask a question, get an answer. It's slower, it's more expensive, but it's *very easy to do*.


**`video/gemini-upload.py`** — Upload a video file to Gemini and ask a question about it


In [ ]:
import time
from pathlib import Path

from pydantic_ai import Agent, VideoUrl
from pydantic_ai.providers.google import GoogleProvider

DATA = Path("data")
VIDEO = DATA / "rDXubdQdJYs.mp4"
PROMPT = "Describe what happens in this video."
MODEL = "google-gla:gemini-2.5-flash"

provider = GoogleProvider()
video_file = provider.client.files.upload(file=str(VIDEO))

while video_file.state.name == "PROCESSING":
    time.sleep(5)
    video_file = provider.client.files.get(name=video_file.name)

agent = Agent(MODEL)
result = agent.run_sync([PROMPT, VideoUrl(url=video_file.uri, media_type=video_file.mime_type)])
print(result.output)


This video is a split-screen compilation from the CNN Presidential Debate held in Atlanta, Georgia, on June 27, 2024, featuring President Joe Biden and former President Donald Trump. The video highlights several contentious exchanges between the two candidates.

Here's a breakdown of what happens:

*   **0:00-0:02:** The video opens with a split screen showing Joe Biden (top) and Donald Trump (bottom) separately walking onto the debate stage, both appearing serious.
*   **0:02-0:07:** Biden is shown speaking about "more border patrol and more asylum officers," with a somewhat hesitant delivery.
*   **0:07-0:12:** Trump responds, looking dismissive, saying, "I really don't know what he said at the end of that sentence. I don't think he knows what he said either."
*   **0:12-0:17:** Biden then directly attacks Trump, stating angrily, "The only person on this stage that is a convicted felon is the man I'm looking at right now."
*   **0:17-0:20:** Trump quickly retorts, pointing a finger, 

## Or just give it a YouTube URL

If the video is already on YouTube, it's even easier: Gemini accepts YouTube URLs directly.


**`video/gemini-youtube.py`** — Send a YouTube URL directly to Gemini for analysis (no download needed)


In [ ]:
from pathlib import Path

from pydantic_ai import Agent, VideoUrl

URL = "https://www.youtube.com/watch?v=rDXubdQdJYs"
PROMPT = "What topics are discussed in this video?"
MODEL = "google-gla:gemini-2.5-flash"

agent = Agent(MODEL)
result = agent.run_sync([
    PROMPT,
    VideoUrl(url=URL),
])

print(result.output)


The video features segments from a presidential debate between Joe Biden and Donald Trump, and the topics discussed include:

*   **Immigration and border security:** Biden mentions increasing border patrol and asylum officers.
*   **Felony convictions:** Biden refers to Trump as a "convicted felon," and Trump retorts by mentioning Hunter Biden's conviction.
*   **Personal insults and character attacks:** Biden calls Trump names like "alley cat," "sucker," and "loser."
*   **The economy and inflation:** Trump blames Biden for inflation.
*   **Mental fitness and cognitive ability:** Trump boasts about passing cognitive tests and implies Biden couldn't, while Biden calls Trump "less competent" and refers to his own record.
*   **Healthcare:** Biden makes a somewhat garbled statement about "beating Medicare" (likely a verbal gaffe meaning improving it).


## Ask about a specific moment

"What's happening at 1:30?" Gemini can jump to timestamps. Useful when you already know *where* to look but need the AI to describe *what* it sees.


**`video/gemini-timestamp.py`** — Ask Gemini about a specific moment in a video using timestamps


In [ ]:
import time
from pathlib import Path

from pydantic_ai import Agent, VideoUrl
from pydantic_ai.providers.google import GoogleProvider

DATA = Path("data")
VIDEO = DATA / "rDXubdQdJYs.mp4"
PROMPT = "What is happening at 01:30 in this video? Describe the scene and any text on screen."
MODEL = "google-gla:gemini-2.5-flash"

provider = GoogleProvider()
video_file = provider.client.files.upload(file=str(VIDEO))

while video_file.state.name == "PROCESSING":
    time.sleep(5)
    video_file = provider.client.files.get(name=video_file.name)

agent = Agent(MODEL)
result = agent.run_sync([PROMPT, VideoUrl(url=video_file.uri, media_type=video_file.mime_type)])
print(result.output)


At 01:30 in the video, Joe Biden is shown in a close-up shot, speaking intensely into a microphone. He is wearing a dark suit, a white shirt, and a blue tie, with his brows furrowed and his mouth slightly open as he speaks.

The background is a solid blue with a faint, repeating CNN logo pattern.

**Text on screen:**

*   **Top left:** "wp" (logo)
*   **Top right:** "Courtesy of CNN" and "Atlanta, Ga. | June 27, 2024"
*   **Bottom center (subtitles):** "My son was not a loser. He was not a sucker. You're the sucker."


## Structured scene-by-scene breakdown

If you join Gemini with Pydantic, you can ask for exactly what you want: timestamps, descriptions, people visible, text on screen. It's the same structured output pattern from the image notebooks, just applied to video.


**`video/gemini-structured.py`** — Get a structured scene-by-scene breakdown of a video


In [ ]:
import time
from pathlib import Path

from pydantic import BaseModel, Field
from pydantic_ai import Agent, VideoUrl
from pydantic_ai.providers.google import GoogleProvider

DATA = Path("data")
VIDEO = DATA / "rDXubdQdJYs.mp4"
MODEL = "google-gla:gemini-2.5-flash"

class Scene(BaseModel):
    start: str = Field(description="Start timestamp MM:SS")
    end: str = Field(description="End timestamp MM:SS")
    description: str = Field(description="What happens in this scene")
    people_visible: list[str] = Field(description="People visible")
    text_on_screen: str = Field(description="Any chyrons, captions, or on-screen text", default="")

provider = GoogleProvider()
video_file = provider.client.files.upload(file=str(VIDEO))

while video_file.state.name == "PROCESSING":
    time.sleep(5)
    video_file = provider.client.files.get(name=video_file.name)

agent = Agent(MODEL, output_type=list[Scene])
result = agent.run_sync([
    "Break this video into scenes. For each scene identify timestamps, "
    "what happens, who is visible, and any text on screen.",
    VideoUrl(url=video_file.uri, media_type=video_file.mime_type),
])
for s in result.output:
    print(f"[{s.start} - {s.end}] {s.description}")
    if s.people_visible:
        print(f"  People: {', '.join(s.people_visible)}")
    if s.text_on_screen:
        print(f"  Text: {s.text_on_screen}")


[00:00 - 00:01] Split screen showing Joe Biden walking on the top and Donald Trump walking on the bottom.
  People: Joe Biden, Donald Trump
  Text: PRESIDENTIAL DEBATE, E PLURIBUS UNUM, wp, CNN (repeated), Courtesy of CNN, Atlanta, Ga. | June 27, 2024
[00:01 - 00:04] Close-up of Joe Biden speaking, with subtitles.
  People: Joe Biden
  Text: ...relative to what we're going to do with, wp, Courtesy of CNN, Atlanta, Ga. | June 27, 2024
[00:04 - 00:06] Close-up of Donald Trump speaking, with subtitles.
  People: Donald Trump
  Text: more border patrol and more asylum officers, wp, Courtesy of CNN, Atlanta, Ga. | June 27, 2024
[00:06 - 00:07] Close-up of Joe Biden speaking, with subtitles.
  People: Joe Biden
  Text: more border patrol and more asylum officers, wp, Courtesy of CNN, Atlanta, Ga. | June 27, 2024
[00:07 - 00:13] Close-up of Donald Trump speaking, with subtitles and his name/title.
  People: Donald Trump
  Text: I really don't know what he said at the end of that sentence., Do